In [11]:
import torch
import json
import networkx as nx
from pyvis.network import Network
from pathlib import Path
from IPython.display import display, HTML

# Define your export directory
EXPORT_DIR = Path("/Users/philippebeliveau/Desktop/Notebook/Orientor_project/ESCO/esco_gnn_ready")

# Load tensors and JSON mappings
edge_index = torch.load(EXPORT_DIR / "edge_index.pt")
features = torch.load(EXPORT_DIR / "node_features.pt")

with open(EXPORT_DIR / "edge_type.json") as f:
    edge_types = json.load(f)
with open(EXPORT_DIR / "node2idx.json") as f:
    node2idx = json.load(f)
with open(EXPORT_DIR / "idx2node.json") as f:
    idx2node = json.load(f)

# Print summary
print("Edge index shape:", edge_index.shape)         # (2, num_edges)
print("Node features shape:", features.shape)         # (num_nodes, 1024)
print("Number of edge types:", len(edge_types))       # Should match num_edges
print("Sample node mapping:", list(node2idx.items())[:3])


Edge index shape: torch.Size([2, 150944])
Node features shape: torch.Size([18162, 1024])
Number of edge types: 150944
Sample node mapping: [('skill::key_1260', 0), ('skill::key_1261', 1), ('skill::key_1262', 2)]


| Item              | Value                       | ✅ Indicates                                  |
| ----------------- | --------------------------- | -------------------------------------------- |
| `edge_index`      | `[2, 150944]`               | 150,944 directed edges in the graph          |
| `node_features`   | `[18162, 1024]`             | 18,162 nodes with BGE embeddings (1024-dim)  |
| `edge_types`      | `150944` elements           | One label for each edge                      |
| `node2idx` sample | `'skill::key_1260': 0` etc. | Named nodes are mapped to consistent indices |


In [8]:
assert edge_index.shape[1] == len(edge_types), "Mismatch between edge count and edge_type list"
assert features.shape[0] == len(node2idx), "Mismatch between node features and node mapping"
print("[PASS] Graph export is structurally valid for GNN input.")

[PASS] Graph export is structurally valid for GNN input.


In [ ]:
import torch
import json
from pathlib import Path
from collections import Counter

# === Paths === #
DATA_DIR = Path("/Users/philippebeliveau/Desktop/Notebook/Orientor_project/ESCO/esco_gnn_ready")
with open(DATA_DIR / "node2idx.json") as f:
    node2idx = json.load(f)
with open(DATA_DIR / "idx2node.json") as f:
    idx2node = json.load(f)
with open(DATA_DIR / "edge_type.json") as f:
    edge_types = json.load(f)

edge_index = torch.load(DATA_DIR / "edge_index.pt")

# === Edge Label Diagnostics === #
label_counter = Counter()
valid_edges = []
invalid_edges = []

for i in range(edge_index.shape[1]):
    src_idx = edge_index[0, i].item()
    tgt_idx = edge_index[1, i].item()
    edge_type = edge_types[i]
    src = idx2node[str(src_idx)]
    tgt = idx2node[str(tgt_idx)]

    if "skill::" in src and "occupation::" in tgt:
        if edge_type in {"essential", "optional"}:
            label_counter[edge_type] += 1
            valid_edges.append((src, tgt, edge_type))
        else:
            invalid_edges.append((src, tgt, edge_type))

# === Summary === #
print("✅ Valid skill→occupation edges found:")
for label in ["essential", "optional"]:
    print(f" - {label}: {label_counter[label]}")

print(f"\n⚠️ Invalid or unknown label edges: {len(invalid_edges)}")
if invalid_edges:
    print("Example invalid edge types:")
    for e in invalid_edges[:5]:
        print(" -", e)



✅ Valid skill→occupation edges found:
 - essential: 64891
 - optional: 58897

⚠️ Invalid or unknown label edges: 0
